## 4. Validación de Hipótesis de la Cátedra
Carga de librerías y lectura del dataset limpio proporcionado por el equipo.

In [1]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import statsmodels.api as sm

df = pd.read_parquet('dataset_limpio.parquet')
df.head()

,product_name,brands,categories,ingredients,nova_group,allergens,energy_kcal,fat_100g,saturated_fat_100g,carbs_100g,...,sodium_100g,contains_gluten,contains_dairy,contains_nuts,contains_soy,contains_eggs,contains_fish,nutriscore_num,ecoscore_num,categoria_macro
0,Sidi Ali,سيدي علي,"en:beverages-and-beverages-preparations, en:be...",OBD1 999 999 1112606 266963207 mb,<NA>,NaN,0.0,0.0,0.0,4.2,...,0.000000,False,False,False,False,False,False,5,<NA>,Bebidas
1,Perly,Perly,"en:dairies, en:fermented-foods, en:fermented-m...","milk cream, cream, sugar, banana, bacteria",3,"en:banana, en:milk",97.0,3.0,0.0,9.4,...,0.000000,False,True,False,False,False,False,<NA>,4,Lácteos
2,اكوافينا,AQUAFINA,"en:beverages-and-beverages-preparations, en:be...",ouverture et avant le : Voir bouteille. après ...,<NA>,NaN,0.0,0.0,0.0,0.0,...,0.000203,False,False,False,False,False,False,5,<NA>,Bebidas
3,Lait,Jaouda,"en:dairies, en:meals, en:milks-liquid-and-powder",Lait frais pasteurisé demi-écrémé à 15g/l de m...,3,en:milk,45.0,1.5,1.0,4.9,...,0.090000,False,True,False,False,False,False,4,2,Lácteos
4,Eau De Source,Cristaline,"en:beverages-and-beverages-preparations, en:be...",Eau de source,1,en:none,0.0,0.0,0.0,0.0,...,0.001100,False,False,False,False,False,False,5,<NA>,Bebidas


### 4.b Asociación entre procesamiento industrial (NOVA) y Nutri-Score
**Hipótesis:** El grado de procesamiento industrial de un alimento está asociado con la calificación nutricional Nutri-Score que recibe, de manera que los productos más procesados presentan calificaciones más desfavorables

* **Variables:** `nova_group` (1 al 4) y `nutriscore_grade` (A a E).
* **Prueba estadística 1:** Prueba de Independencia Chi-cuadrado ($\chi^2$) sobre una tabla de contingencia, para confirmar si existe dependencia estadística global entre ambas variables.
* **Prueba estadística 2:** Coeficiente de Correlación de Spearman. Al ser dos variables con un orden jerárquico inherente, esta métrica es ideal para confirmar si la relación es monótona direccional (a mayor número NOVA, peor letra de Nutri-Score).

In [3]:
import pandas as pd
from scipy import stats

df_4b = df[['nova_group', 'nutriscore_num']].dropna().copy()

tabla_contingencia = pd.crosstab(df_4b['nova_group'], df_4b['nutriscore_num'])
chi2, p_chi2, dof, expected = stats.chi2_contingency(tabla_contingencia)

coef_spearman, p_spearman = stats.spearmanr(df_4b['nova_group'], df_4b['nutriscore_num'])

print("--- PRUEBA DE INDEPENDENCIA CHI-CUADRADO ---")
print(f"Estadístico Chi2: {chi2:.4f}")
print(f"P-valor Chi2: {p_chi2:.4e}\n")

print("--- CORRELACIÓN DE SPEARMAN ---")
print(f"Coeficiente de Spearman: {coef_spearman:.4f}")
print(f"P-valor Spearman: {p_spearman:.4e}\n")

print("--- TABLA DE CONTINGENCIA (OBSERVACIONES REALES) ---")
print(tabla_contingencia)

--- PRUEBA DE INDEPENDENCIA CHI-CUADRADO ---
Estadístico Chi2: 854.2177
P-valor Chi2: 3.8689e-175

--- CORRELACIÓN DE SPEARMAN ---
Coeficiente de Spearman: -0.3094
P-valor Spearman: 1.0264e-91

--- TABLA DE CONTINGENCIA (OBSERVACIONES REALES) ---
nutriscore_num    1    2    3    4    5
nova_group                             
1                 7   24   75  107  257
2                83   19   18   46    3
3               127  256  288  161  214
4               682  553  639  284  258


### Conclusión de la Hipótesis 4.b

Los resultados de las pruebas estadísticas confirman la hipótesis: el grado de procesamiento industrial de un producto y su calificación Nutri-Score están directamente asociados.

* **Dependencia (Chi-cuadrado):** El p-valor nos dio ínfimo (3.8689e-175). Esto nos permite descartar de lleno que las variables sean independientes; existe una asociación estadísticamente significativa entre el grupo NOVA y la nota del Nutri-Score.
* **Dirección de la relación (Spearman):** Elegimos usar la correlación de Spearman porque es la métrica ideal cuando trabajamos con variables que tienen un orden jerárquico. El coeficiente nos dio negativo (-0.3094) y altamente significativo (p-valor 1.0264e-91). Esto nos marca una tendencia clara: a medida que un alimento es más ultraprocesado (sube en la escala NOVA), su calificación Nutri-Score tiende a empeorar (cayendo en las letras D y E).